# Tutorial 06: Physical Verification of `Dirac-Solver`

> **Language / Idioma:** [Español](../es/06_verificacion_fisica.ipynb) | **English**

This notebook validates the numerical accuracy and physical conservation laws of the C++ FDTD Leapfrog solver:
1. **Free plane-wave phase evolution:** $\psi(x, t) = \psi(x, 0) e^{-i E t}$ with $E = \sqrt{p_x^2 + m^2}$.
2. **Conservation of total probability:** $P(t) = \int \psi^\dagger \psi \, dV = \text{const.}$
3. **Conservation of expected energy:** $\langle H \rangle(t) = \int \psi^\dagger \hat{H} \psi \, dV = \text{const.}$

In [ ]:
import sys
sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from dirac_solver import (
    Grid, ConstantSpinor, PlaneWave, GaussianPacket,
    DiracProblemBuilder, DiracSolver, FreeParticle, Observables, _core
)

## 1. Free-Particle Plane Wave Phase Evolution

In [ ]:
L, N = 10.0, 200
dx = L / N
grid = Grid(shape=(N,), spacing=(dx,))
print(f'Courant limit dt_max = {grid.courant_limit():.5f}')

px, mass = 1.0, 1.0
spinor = ConstantSpinor(momentum=[px, 0.0, 0.0], mass=mass)
initial_state = PlaneWave(spinor)

dt, total_time = 0.0005, 0.2
problem = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(initial_state)
    .set_potential(FreeParticle())
    .set_boundary_condition(_core.PeriodicBoundary())
    .set_time_parameters(time_step=dt, total_time=total_time)
    .build()
)

solver = DiracSolver(problem)
psi_0 = solver.get_psi().copy()
solver.run_simulation()
psi_t = solver.get_psi()

E = np.sqrt(px**2 + mass**2)
psi_analytical = psi_0 * np.exp(-1j * E * total_time)
rel_error = np.linalg.norm(psi_t - psi_analytical) / np.linalg.norm(psi_analytical)
print(f'Relative L2 error vs. analytical solution: {rel_error:.6e}')

## 2. Conservation of Total Probability and Expected Energy

In [ ]:
grid_gp = Grid(shape=(200,), spacing=(0.1,))
spinor_gp = ConstantSpinor(momentum=[1.5, 0.0, 0.0], mass=1.0)
packet = GaussianPacket(spinor_gp, center=[0.0, 0.0, 0.0], spatial_width=1.5)

problem_gp = (
    DiracProblemBuilder()
    .set_grid(grid_gp)
    .set_initial_state(packet)
    .set_potential(FreeParticle())
    .set_boundary_condition(_core.PeriodicBoundary())
    .set_time_parameters(time_step=0.001, total_time=1.0)
    .build()
)

solver_gp = DiracSolver(problem_gp)
obs = Observables(solver_gp)
solver_gp.run_simulation(observables_handler=obs)

times = obs.get_history('time')
rho_hist = obs.get_history('probability_density')
prob_total = np.sum(rho_hist, axis=1) * grid_gp.get_cell_volume()
energy_hist = obs.get_history('expected_energy')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, prob_total / prob_total[0])
axes[0].set_title('Normalized Total Probability $P(t)/P(0)$')
axes[0].set_xlabel('Time $t$')
axes[0].grid(True)

axes[1].plot(times, energy_hist)
axes[1].set_title('Expected Energy $\\langle H \\rangle(t)$')
axes[1].set_xlabel('Time $t$')
axes[1].grid(True)
plt.tight_layout()
plt.show()

print(f'Max relative probability drift: {np.max(np.abs(prob_total - prob_total[0])) / prob_total[0]:.4e}')
print(f'Max relative energy drift:      {np.max(np.abs(energy_hist - energy_hist[0])) / abs(energy_hist[0]):.4e}')